# lerobot-compare quickstart: comparing two policies

This notebook shows the whole workflow end to end:

1. **Build a test dataset** of paired rollouts (here synthetic, so it runs anywhere with no GPU).
2. **Run a comparison** with each of the two test methods.
3. **Read the result** — a scorecard modelled on experimentation platforms such as [Statsig](https://statsig.com).

**A/B test** = a controlled experiment that compares variant **A** against variant **B**. Here A and B are two robot/agent policies and the metric is task **success rate**.

> Install: `pip install -e ".[viz]"` (the core needs only numpy + scipy; `viz` adds matplotlib for the plots in notebook&nbsp;2).

## 1. Build a synthetic test dataset

We pair each scene so the same situation is attempted by both policies — the lowest-variance way to compare them.

In [1]:
import numpy as np
from lerobot_doctor import Trial

def make_dataset(rate_A, rate_B, n, seed=0):
    """Build a synthetic *paired* dataset for an A/B comparison.

    Each of `n` scenes is attempted by policy A and policy B. A `Trial` records a
    0/1 success and a scene `key`; sharing the key lets lerobot-compare form a clean
    *paired* (exact-match) test, which has lower variance than comparing the two
    policies as independent samples.

    In a real project you would replace these Bernoulli draws with your own
    per-rollout success labels (a human's yes/no, or a calibrated judge).
    """
    rng = np.random.default_rng(seed)
    A = [Trial(success=int(rng.random() < rate_A), key=k) for k in range(n)]
    B = [Trial(success=int(rng.random() < rate_B), key=k) for k in range(n)]
    return A, B

# Policy A succeeds 65% of the time, policy B 55%, over 800 paired scenes.
A, B = make_dataset(rate_A=0.65, rate_B=0.55, n=800, seed=0)
print(f"built {len(A)} paired trials; "
      f"A successes = {sum(t.success for t in A)}, "
      f"B successes = {sum(t.success for t in B)}")


built 800 paired trials; A successes = 499, B successes = 433


## 2. Run a comparison

`compare_success(A, B, method=...)` is the single entry point. Two methods ship today; their full, descriptive names say exactly what each is:

In [2]:
from lerobot_doctor import compare_success, available_methods

available_methods()

['anytime-valid-betting-e-process', 'fixed-horizon-proportion-z-test']

### Method 1 — anytime-valid betting e-process (the default)

**Anytime-valid** means the error guarantee holds at *every* sample size, so you may watch the result live and stop the moment it is conclusive, with no statistical penalty for peeking. It reports an **e-value** (evidence against “no difference”) instead of a p-value.

In [3]:
report_av = compare_success(A, B, alpha=0.05)   # default method
print(report_av.summary())

lerobot-compare | A/B result -- anytime-valid / betting e-process, 95% confidence
--------------------------------------------------------------------------------
  Decision: NO CALL yet  (e-value 1.4 < 40; need more rollouts)

  Variant      Success rate              95% CI (anytime-valid)
  A (test)     62.4%  (499/800)          [58.9%, 65.7%]
  B (control)  54.1%  (433/800)          [50.6%, 57.6%]

  Absolute lift (A-B):  +8.2 pts   95% CI [-1.2, +16.3] (anytime-valid; widened)
  Relative lift:        +15.2%   95% CI [-2.3%, +30.0%]
  Bayesian P(A>B):      99.9%
  Health:               sample-ratio OK (n_A=800, n_B=800, SRM p=1)
  Match:                exact (clean paired test)
  Rungs:                {'exact': 800, 'covariate': 0, 'unpaired': 0}


### Method 2 — fixed-horizon proportion z-test (the classic A/B test)

**Fixed-horizon** means you fix the sample size in advance and analyse **once**. This is the textbook test most platforms run by default. It reports a **p-value**, a **standard error (SE)**, the achieved statistical **power**, and the **minimum detectable effect (MDE)**.

In [4]:
report_fx = compare_success(A, B, method="fixed-horizon-proportion-z-test", alpha=0.05)
print(report_fx.summary())

lerobot-compare | A/B result -- fixed-horizon / proportion z-test, 95% confidence
--------------------------------------------------------------------------------
  Decision: A>B is SIGNIFICANT  (p = 0.0009819)

  Variant      Success rate              95% CI
  A (test)     62.4%  (499/800)          [58.9%, 65.7%]
  B (control)  54.1%  (433/800)          [50.6%, 57.6%]

  Absolute lift (A-B):  +8.2 pts   95% CI [+3.3, +13.2]
  Relative lift:        +15.2%   95% CI [+6.2%, +24.3%]
  Standard error:       0.025
  Bayesian P(A>B):      99.9%
  Health:               sample-ratio OK (n_A=800, n_B=800, SRM p=1)
  Power:                0.91 achieved; MDE at 80% power = 7.0 pts
  Match:                exact (clean paired test)
  Rungs:                {'exact': 800, 'covariate': 0, 'unpaired': 0}


## 3. Read the result programmatically

Everything in the scorecard is also a field on the returned `Report`, so you can log it, gate CI on it, or serialise it to JSON.

| Field | Meaning |
|---|---|
| `decided` / `direction` | did we call a winner, and which way |
| `effect` | absolute lift = `rate_A - rate_B` (in proportion units) |
| `effect_ci_lo/.._hi` | confidence interval (**CI**) on that lift |
| `effect_relative` | relative lift = `(rate_A - rate_B) / rate_B` (Statsig's “Delta %”) |
| `p_value` | fixed-horizon p-value (`None` for the e-process) |
| `e_value` / `e_threshold` | e-process evidence and the bar it must clear (`None` for fixed) |
| `chance_to_beat` | Bayesian posterior P(A beats B) |
| `srm_p_value` / `srm_flag` | Sample Ratio Mismatch health check |

In [5]:
r = report_fx
print(f"decided      : {r.decided}  (direction {r.direction})")
print(f"absolute lift: {r.effect:+.3f}  CI [{r.effect_ci_lo:+.3f}, {r.effect_ci_hi:+.3f}]")
print(f"relative lift: {r.effect_relative:+.1%}")
print(f"p-value      : {r.p_value:.4g}")
print(f"std error    : {r.standard_error:.4f}")
print(f"P(A beats B) : {r.chance_to_beat:.1%}")
print(f"SRM p-value  : {r.srm_p_value:.3f}  (flagged: {r.srm_flag})")

decided      : True  (direction A>B)
absolute lift: +0.083  CI [+0.033, +0.132]
relative lift: +15.2%
p-value      : 0.0009819
std error    : 0.0250
P(A beats B) : 99.9%
SRM p-value  : 1.000  (flagged: False)


### One-sided tests and JSON export

Use `alternative="A>B"` when you only care whether A *beats* B (not whether they merely differ). `to_dict()` gives a JSON-serialisable record.

In [6]:
import json
one_sided = compare_success(A, B, alternative="A>B", alpha=0.05)
print("one-sided decided:", one_sided.decided, one_sided.direction)

blob = json.dumps(report_fx.to_dict())
print("serialised report is", len(blob), "bytes of JSON")

one-sided decided: False None
serialised report is 11101 bytes of JSON


## Where next

- **[02_fixed_vs_anytime_valid.ipynb](02_fixed_vs_anytime_valid.ipynb)** — how the two methods differ as the effect size changes, and why the e-process's advantage is *early stopping*, not more power at a fixed sample size.
- For the **label-efficient** mode (a cheap judge on every rollout, scarce human checks only where they matter), see the *Label-efficient mode* section of the README.